# Predict — Loan Amount Inference

Run the trained model on a single borrower profile. Adjust the inputs in the cell below.

In [ ]:
import os, sys, pickle
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf

with open('../outputs/models/scaler_X.pkl','rb') as f: scaler_X = pickle.load(f)
with open('../outputs/models/scaler_y.pkl','rb') as f: scaler_y = pickle.load(f)
with open('../outputs/models/linear_regression.pkl','rb') as f: lr = pickle.load(f)
nn = tf.keras.models.load_model('../outputs/models/best_nn.keras')

feature_names = scaler_X.feature_names_in_
print('Models loaded.')

## Define a Borrower Profile

Edit these values. Valid ranges: age 20–65, income $10k–$500k, emp_length 0–41 yrs, int_rate 5.4–23.2%, cred_hist 2–30 yrs.

In [ ]:
# ── Edit these ───────────────────────────────────────────────────────────────
person_age             = 28
person_income          = 55000
person_emp_length      = 3
loan_int_rate          = 11.5
cb_person_cred_hist    = 4

person_home_ownership  = 'RENT'       # RENT | MORTGAGE | OWN | OTHER
loan_intent            = 'PERSONAL'   # PERSONAL | EDUCATION | MEDICAL | VENTURE | HOMEIMPROVEMENT | DEBTCONSOLIDATION
loan_grade             = 'B'          # A | B | C | D | E | F | G
cb_person_default      = 'N'          # Y | N
# ─────────────────────────────────────────────────────────────────────────────

raw = [
    person_age, person_income, person_emp_length, loan_int_rate, cb_person_cred_hist,
    1.0 if person_home_ownership == 'OTHER' else 0.0,
    1.0 if person_home_ownership == 'OWN'   else 0.0,
    1.0 if person_home_ownership == 'RENT'  else 0.0,
    1.0 if loan_intent == 'EDUCATION'        else 0.0,
    1.0 if loan_intent == 'HOMEIMPROVEMENT'  else 0.0,
    1.0 if loan_intent == 'MEDICAL'          else 0.0,
    1.0 if loan_intent == 'PERSONAL'         else 0.0,
    1.0 if loan_intent == 'VENTURE'          else 0.0,
    1.0 if loan_grade == 'B' else 0.0,
    1.0 if loan_grade == 'C' else 0.0,
    1.0 if loan_grade == 'D' else 0.0,
    1.0 if loan_grade == 'E' else 0.0,
    1.0 if loan_grade == 'F' else 0.0,
    1.0 if loan_grade == 'G' else 0.0,
    1.0 if cb_person_default == 'Y' else 0.0,
]

X_input = pd.DataFrame([raw], columns=feature_names)
X_scaled = scaler_X.transform(X_input)
print('Input ready.')

## Run Predictions

In [ ]:
# Neural network
nn_scaled = nn.predict(X_scaled, verbose=0).flatten()[0]
nn_usd    = scaler_y.inverse_transform([[nn_scaled]])[0][0]

# Linear regression
lr_usd = lr.predict(X_scaled)[0]

print(f'Neural Network:     ${nn_usd:,.0f}')
print(f'Linear Regression:  ${lr_usd:,.0f}')

## Compare Multiple Profiles

In [ ]:
profiles = [
    dict(age=22, income=30000,  emp=1,  rate=14.0, hist=2,  home='RENT',     intent='EDUCATION',     grade='C', default='N'),
    dict(age=35, income=90000,  emp=8,  rate=8.5,  hist=10, home='MORTGAGE', intent='HOMEIMPROVEMENT',grade='A', default='N'),
    dict(age=45, income=150000, emp=15, rate=7.0,  hist=20, home='OWN',      intent='DEBTCONSOLIDATION',grade='A',default='N'),
    dict(age=28, income=45000,  emp=2,  rate=16.5, hist=3,  home='RENT',     intent='MEDICAL',       grade='D', default='Y'),
]

rows = []
for p in profiles:
    r = [
        p['age'], p['income'], p['emp'], p['rate'], p['hist'],
        1.0 if p['home']=='OTHER' else 0.0, 1.0 if p['home']=='OWN' else 0.0, 1.0 if p['home']=='RENT' else 0.0,
        1.0 if p['intent']=='EDUCATION' else 0.0, 1.0 if p['intent']=='HOMEIMPROVEMENT' else 0.0,
        1.0 if p['intent']=='MEDICAL' else 0.0, 1.0 if p['intent']=='PERSONAL' else 0.0,
        1.0 if p['intent']=='VENTURE' else 0.0,
        1.0 if p['grade']=='B' else 0.0, 1.0 if p['grade']=='C' else 0.0,
        1.0 if p['grade']=='D' else 0.0, 1.0 if p['grade']=='E' else 0.0,
        1.0 if p['grade']=='F' else 0.0, 1.0 if p['grade']=='G' else 0.0,
        1.0 if p['default']=='Y' else 0.0,
    ]
    rows.append(r)

X_batch = pd.DataFrame(rows, columns=feature_names)
X_batch_sc = scaler_X.transform(X_batch)

nn_preds = scaler_y.inverse_transform(nn.predict(X_batch_sc, verbose=0)).flatten()
lr_preds = lr.predict(X_batch_sc)

results = pd.DataFrame({
    'Age':    [p['age']    for p in profiles],
    'Income': [f'${p["income"]:,}' for p in profiles],
    'Grade':  [p['grade']  for p in profiles],
    'Intent': [p['intent'] for p in profiles],
    'NN ($)': [f'${v:,.0f}'  for v in nn_preds],
    'LR ($)': [f'${v:,.0f}'  for v in lr_preds],
})
results